# ML-пайплайн: отбор и категоризация мемов

В этом ноутбуке разобран процесс подготовки данных для приложения «Погода с настроением»: загрузка датасета мемов, отбор погодных мемов по ключевым словам и zero-shot категоризация новых мемов с помощью эмбеддингов.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / 'backend').exists():
    sys.path.insert(0, str(ROOT / 'backend'))
elif (ROOT.parent / 'backend').exists():
    sys.path.insert(0, str(ROOT.parent / 'backend'))


## 1. Данные

Датасет `foldl/rumeme-desc` (Hugging Face) — 2508 русских мемов с колонками `image` и `text`.

In [ ]:
from app.ml.fetch_memes import load_table

table = load_table()
print('rows:', table.num_rows)
print('schema:', table.schema)


Примеры описаний (первые 5, обрезаны до 160 символов):

In [ ]:
texts = table.column('text').to_pylist()
for i in range(5):
    print(f'[{i}] {texts[i][:160]}')


## 2. Отбор погодных мемов по ключевым словам

Датасет общий, погодных мемов меньшинство. Для seed используем детерминированный отбор по ключевым словам со списком исключений (например, «ветеринар» не должен попадать в «ветер»).

In [ ]:
from app.ml.categorize import CATEGORY_KEYWORDS, categorize_by_keywords

print('Keywords:', CATEGORY_KEYWORDS)
matched = []
for i, text in enumerate(texts):
    cat = categorize_by_keywords(text)
    if cat:
        matched.append((i, cat))
print('matched:', len(matched), '/', len(texts))
print('distribution:', dict(Counter(c for _, c in matched)))


## 3. Эмбеддинги (fastembed)

Для семантической категоризации используем мультиязычные эмбеддинги. Проверяем, что модель даёт осмысленную близость между якорями категорий.

In [ ]:
import numpy as np
from app.ml.categorize import ANCHORS, CATEGORIES
from app.ml.embedding import build_embedder

anchors = [ANCHORS[c] for c in CATEGORIES]
embedder = build_embedder(anchors)
vectors = embedder.encode(anchors)
sim = vectors @ vectors.T

print('Cosine similarity matrix (categories):')
print('       ' + '  '.join(f'{c:>7}' for c in CATEGORIES))
for i, c in enumerate(CATEGORIES):
    row = '  '.join(f'{sim[i, j]:.2f}' for j in range(len(CATEGORIES)))
    print(f'{c:>7} {row}')


## 4. Zero-shot категоризация (для пользовательских мемов)

Для мемов, добавленных пользователями без категории, категория определяется по близости описания к якорям категорий.

In [ ]:
from app.ml.categorize import categorize

samples = [
    'сегодня весь день льёт дождь, а зонт забыл дома',
    'выпал первый снег и сразу сугробы',
    'на улице жара +35 и солнце печёт',
]
print('samples ->', categorize(samples))


## 5. Итог

Полученная выборка записывается в базу данных через `app.ml.pipeline`. Дальше рекомендация мемов внутри категории выполняется онлайн-бандитом (Thompson sampling) по оценкам пользователей.